# Week 12 Lab: The Professional ML Workflow

<a href="https://colab.research.google.com/github/bradleyboehmke/uc-bana-4080/blob/main/labs/lab-12-model-tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this lab, you'll stop peeking at the test set for good. You'll score models with cross-validation, tune them with `GridSearchCV`, and put feature engineering inside a pipeline so nothing leaks. Then you'll run the complete professional workflow from start to finish on the Ames housing data.

## 🎯 Learning Objectives
By the end of this lab, you will be able to:
- Use k-fold cross-validation to compare models and settings without touching the test set
- Read a CV score's mean and standard deviation
- Tune hyperparameters systematically with `GridSearchCV`
- Build a `ColumnTransformer` + `Pipeline` that scales and encodes features without data leakage
- Run the complete workflow: split, make every decision with CV, refit, then test once

## 📚 This Lab Reinforces
- **Chapter 28: Cross-Validation**
- **Chapter 29: Hyperparameter Tuning**
- **Chapter 30: Feature Engineering**
- **Week 12 Tuesday Slides: The Professional ML Workflow**

## 🕐 Estimated Time & Structure
**Total Time:** 75 minutes
**Mode:** Work together in groups (2–4 students), submit individually

- **[0–25 min]** Guided review of Tuesday's workflow on the Ames housing data
- **[25–30 min]** Class discussion and Q&A
- **[30–70 min]** Independent challenges: the complete workflow on a new set of Ames features
- **[70–75 min]** Lab wrap-up and homework quiz preparation

You are encouraged to work through the lab with **2–4 classmates**: talk through the problems and help each other get past coding hurdles. However, **you must write your own code, and you will submit your own notebook and homework quiz.**

## 💡 Why This Matters
When you tell a manager "this model will be off by about $30K per home," that number has to be honest. Every time a test score helps make a decision, it stops being honest. Cross-validation, systematic tuning and leak-free pipelines are how professionals make every decision on the training data and keep the test score for one final, trustworthy check.

## Setup

We'll use the **Ames housing** data for the whole lab. Each row is a home sold in Ames, Iowa, and `SalePrice` is what we'll predict. The Guided review uses the same features as Tuesday's lecture; the Challenges use a different set.

Everything we need is already installed in Colab. Run both cells below.

In [ ]:
# Required imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

# Use this random state everywhere so your results match the homework quiz
RANDOM_STATE = 42

pd.set_option('display.max_columns', None)
print("✅ All libraries imported successfully!")

In [ ]:
# Load the Ames housing data (local copy if you have one, otherwise from GitHub)
try:
    ames = pd.read_csv('../data/ames_clean.csv')
except FileNotFoundError:
    url = 'https://raw.githubusercontent.com/bradleyboehmke/uc-bana-4080/main/data/ames_clean.csv'
    ames = pd.read_csv(url)

print(f"Dataset shape: {ames.shape}")
ames.head()

## Part A — Guided Review: Tuesday's Workflow on Ames (25 minutes)

Let's rerun Tuesday's workflow, then change a few settings to see how the results respond. The code matches the lecture (and the 💻 Tuesday Your Turn Notebook), so you can look back at either one.

**📋 Workflow:**
1. Split the data, then cross-validate a decision tree
2. Tune the tree with `GridSearchCV`
3. See why KNN needs scaling, using a `Pipeline`
4. Put scaling and encoding together in a `ColumnTransformer` pipeline

Run each cell, and when you see 🔧 **Try it**, change the values and re-run the cell.

### Step 1: Cross-Validate a Tree

First, split the data and lock the test set away. From here until the final evaluation, **every decision uses only the training data**.

In [ ]:
# Step 1a: Split first, then lock the test set away
features = ['GrLivArea', 'LotArea', 'YearRemodAdd', 'TotRmsAbvGrd', 'GarageArea', 'Fireplaces']
X = ames[features]
y = ames['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

print(f"Training set: {len(X_train)} homes")
print(f"Test set:     {len(X_test)} homes (🔒 locked)")

In [ ]:
# Step 1b: 5-fold CV on the TRAINING set only
tree_depth = 6   # 🔧 change this value and re-run the cell

tree = DecisionTreeRegressor(max_depth=tree_depth, random_state=RANDOM_STATE)
cv_scores = cross_val_score(tree, X_train, y_train, cv=5,
                            scoring='neg_root_mean_squared_error')
cv_rmse = -cv_scores   # scikit-learn reports negative RMSE so that higher is always better

print(f"max_depth = {tree_depth}")
print(f"RMSE per fold: {cv_rmse.round()}")
print(f"CV RMSE: ${cv_rmse.mean():,.0f} ± ${cv_rmse.std():,.0f}")

**Reading the output:**
- **The minus sign.** scikit-learn's scorers treat *higher as better*, so error metrics come back negative. Flip the sign to get ordinary RMSE.
- **The mean** (about $51K for a depth-6 tree) is our estimate of how this tree will do on new homes.
- **The std** (about $8K) shows how much the score moves depending on which homes are held out. The five folds range from about $39K to $62K: any one of them, on its own, could have badly misled you.

🔧 **Try it (3 minutes):** Re-run the cell above with `tree_depth` set to **2, 4, 8, and `None`** (no limit). Note the mean and std for each.
- Which depth has the lowest CV RMSE?
- What happens to the **std** as the tree gets deeper? What does that tell you?

*Your notes:*

### Step 2: Tune the Tree with GridSearchCV

You probably found that **depth 4** scored best (about $49K), and that deeper trees did worse *and* got less consistent: their std climbed to about $9K as they started memorizing the training folds.

Testing one depth at a time by hand gets tedious fast, especially once a second setting like `min_samples_leaf` joins in. `GridSearchCV` tries **every combination** in a grid, scores each with cross-validation on the training set, and keeps the best. Here's the search from class: 6 depths × 5 leaf sizes = 30 settings, each with 5-fold CV.

In [ ]:
# Step 2a: The grid search from class
param_grid = {
    'max_depth': [2, 4, 6, 8, 10, None],
    'min_samples_leaf': [1, 5, 10, 20, 40],
}
grid_search = GridSearchCV(
    DecisionTreeRegressor(random_state=RANDOM_STATE),
    param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,                       # use all CPU cores
)
grid_search.fit(X_train, y_train)    # 30 settings × 5 folds = 150 models

print("Best settings:", grid_search.best_params_)
print(f"Best CV RMSE: ${-grid_search.best_score_:,.0f}")

In [ ]:
# Step 2b: The five best cells of the grid
(
    pd.DataFrame(grid_search.cv_results_)
    .assign(cv_rmse=lambda df: -df['mean_test_score'])
    [['param_max_depth', 'param_min_samples_leaf', 'cv_rmse']]
    .sort_values('cv_rmse')
    .head()
)

**What you get back:**
- `best_params_`: the winning cell of the grid. Requiring at least 10 homes per leaf let a depth-6 tree beat every depth on its own, at about $44.7K.
- `best_score_`: its CV score (negative RMSE again, so flip the sign)
- `best_estimator_`: the winning setting **refit on all of `X_train`**, ready to test once at the very end
- `cv_results_`: the score of **every** cell, as in the table above

### 🧪 Practice Exercise — Tune a Decision Tree

**Business Scenario:** You're building a decision tree for your real estate company's pricing tool, and you want to see whether adding a third setting, `min_samples_split`, finds an even better tree.

**Your Task:** Fill in the `___` blanks to tune a `DecisionTreeRegressor` across these values:
- `max_depth`: [5, 10, 15, 20]
- `min_samples_split`: [2, 5, 10]
- `min_samples_leaf`: [1, 2, 4]

Then answer: Did your search beat the 30-setting search above? If not, why might a grid with *more* combinations still lose?

In [ ]:
# Step 1: Define the parameter grid
practice_grid = {
    'max_depth': ___,
    'min_samples_split': ___,
    'min_samples_leaf': ___,
}

# Step 2: Create the DecisionTreeRegressor
dt = DecisionTreeRegressor(random_state=RANDOM_STATE)

# Step 3: Create GridSearchCV
practice_search = GridSearchCV(
    dt,
    ___,
    cv=___,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

# Step 4: Fit on the training data
practice_search.fit(___, ___)

# Step 5: Print the best parameters and best CV RMSE
print("Best parameters:", practice_search.___)
print(f"Best CV RMSE: ${-practice_search.___:,.0f}")

*Your answer:*

### Step 3: Scaling Matters for KNN

KNN predicts a home's price from its nearest neighbors, and "nearest" is measured by distance. Look at how differently our features are spread out:

In [ ]:
# Step 3a: How spread out is each feature?
X_train.agg(['mean', 'std']).round()

`LotArea` has a standard deviation of over 10,000 square feet, while `TotRmsAbvGrd` and `Fireplaces` vary by a couple of units. On raw features, distance is almost entirely lot size. `StandardScaler` puts every feature in standard deviations from its mean, so each one gets a fair say.

Put the scaler **inside a pipeline**, so that in every CV fold it learns its means and stds only from that fold's training rows:

In [ ]:
# Step 3b: The same KNN model on raw features vs. scaled features (scaler inside a pipeline)
k = 10   # 🔧 change this value and re-run the cell

models = {
    'Raw features': KNeighborsRegressor(n_neighbors=k),
    'Scaled (in a pipeline)': Pipeline([('scale', StandardScaler()),
                                        ('model', KNeighborsRegressor(n_neighbors=k))]),
}
for name, model in models.items():
    cv_rmse = -cross_val_score(model, X_train, y_train, cv=5,
                               scoring='neg_root_mean_squared_error')
    print(f"{name:24s} CV RMSE: ${cv_rmse.mean():,.0f}")

🔧 **Try it (2 minutes):** Re-run the cell with `k` set to **5 and 25**. Does scaling help at every `k`? By roughly how much?

*Your notes:*

**🎯 When does scaling matter?**

Scaling helped KNN by about $10K of RMSE. It doesn't help every model, though:

| Model Type                             | Does Scaling Change Performance? | Why                                         |
| -------------------------------------- | -------------------------------- | ------------------------------------------- |
| **LinearRegression (OLS)**             | ❌ No                             | Closed-form solution, not iterative         |
| **Ridge / Lasso / ElasticNet**         | ✅ Yes                            | Regularization depends on coefficient sizes |
| **Logistic Regression (solver-based)** | ✅ Yes                            | Gradient descent & regularization           |
| **SVM / SVR**                          | ✅ Yes                            | Distance-based kernel computations          |
| **KNN / K-means / PCA / clustering**   | ✅ Yes                            | Distance/variance sensitive                 |
| **Neural networks**                    | ✅ Yes                            | Gradient descent stability                  |
| **Tree models (decision trees, random forests)** | ❌ No                   | Trees are scale-invariant                   |

### Step 4: A ColumnTransformer Pipeline

Real data mixes numbers and categories, and each kind needs different treatment. A `ColumnTransformer` gives each group of columns its own step:
- **Numeric columns** → `StandardScaler()`
- **Categories with no order** (house style, zoning) → `OneHotEncoder(handle_unknown='ignore')`, one 0/1 column per category
- **Categories with a real order** (kitchen and exterior quality: Fair < Typical < Good < Excellent) → `OrdinalEncoder` with the order spelled out

A `Pipeline` glues that preprocessing to the model, so `cross_val_score` refits **everything** in each fold. This is the pattern you'll use in the Challenges.

In [ ]:
# Step 4a: Scale, one-hot encode and ordinal-encode inside one pipeline
numeric = ['GrLivArea', 'LotArea', 'YearRemodAdd', 'TotRmsAbvGrd', 'GarageArea', 'Fireplaces']
categorical = ['HouseStyle', 'MSZoning']
quality = ['KitchenQual', 'ExterQual']
quality_order = ['Fa', 'TA', 'Gd', 'Ex']   # Fair < Typical < Good < Excellent

preprocess = ColumnTransformer([
    ('scale', StandardScaler(), numeric),                                   # numbers → scaled
    ('one_hot', OneHotEncoder(handle_unknown='ignore'), categorical),       # no order → 0/1 columns
    ('ordinal', OrdinalEncoder(categories=[quality_order] * 2), quality),   # real order → 0, 1, 2, 3
])
knn_pipeline = Pipeline([('prep', preprocess),
                         ('model', KNeighborsRegressor(n_neighbors=10))])

X_train, X_test, y_train, y_test = train_test_split(
    ames[numeric + categorical + quality], ames['SalePrice'], test_size=0.2, random_state=RANDOM_STATE)
cv_rmse = -cross_val_score(knn_pipeline, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error')
print(f"CV RMSE: ${cv_rmse.mean():,.0f}")

Adding the categories and quality ratings drops KNN's CV RMSE to about $36K.

Because a pipeline acts like a model, `GridSearchCV` can tune it too. To reach a setting *inside* a step, write **the step's name**, **two underscores**, then **the parameter**: `model__n_neighbors` means "the `n_neighbors` of the step we named `model`".

In [ ]:
# Step 4b: Tune a setting inside the pipeline: step name + __ + parameter
param_grid = {'model__n_neighbors': [3, 5, 7, 10, 15, 25]}
pipe_search = GridSearchCV(knn_pipeline, param_grid, cv=5,
                           scoring='neg_root_mean_squared_error', n_jobs=-1)
pipe_search.fit(X_train, y_train)

print("Best settings:", pipe_search.best_params_)
print(f"Best CV RMSE: ${-pipe_search.best_score_:,.0f}")

**💡 Why pipelines are essential:**
- ✅ **No leakage:** the scaler and encoders learn only from each fold's training rows, never from the validation fold or the test set
- ✅ **The preprocessing travels with the model:** new data gets exactly the same treatment when you call `.predict()`
- ✅ **One object to tune:** `GridSearchCV` can search preprocessing and model settings together

## Class Discussion/Q&A (5 minutes)

**Discussion prompts:**
- As your tree got deeper, its CV std grew. Why is a big std a warning sign, even when the mean looks fine?
- Why do we choose `max_depth` with cross-validation instead of picking whichever depth scores best on the test set?
- Scaling cut KNN's error by about $10K, yet the table says it wouldn't change a random forest's at all. What is it about how a tree splits that makes it ignore scale?
- Why must the scaler live *inside* the pipeline, instead of being fit on all of `ames` before splitting?

**Common blockers and clarifications:**
- scikit-learn reports error scores as **negative** numbers (`neg_root_mean_squared_error`); flip the sign to get RMSE
- Inside a pipeline, grid keys are `stepname__parameter`, like `'model__n_neighbors'`; `pipeline.get_params().keys()` lists every valid name
- `n_jobs=-1` runs the work on every CPU core. A cell showing `[*]` is still running, not frozen
- `handle_unknown='ignore'` stops `OneHotEncoder` from crashing when a validation fold or the test set contains a category its training rows never saw

## Part B — Independent Challenges (40 minutes)

Now it's time to apply the complete professional ML workflow to the Ames data. The specific numerical results you get will be used to answer questions on this week's **Canvas homework quiz**.

* **The support decreases as you go.** Challenges 1–2 give you starter code with blanks (🧩), Challenges 3–4 give you an outline only (📝), and Challenges 5–6 are on your own (🚀).
* **Work together, submit individually.** Talk through the challenges with your group and help each other get unstuck, but **write your own code**. You'll each submit your own notebook and quiz.
* **DO NOT USE AI** to generate code for you.
* Feel free to ask questions or seek help from the instructor. Periodically, we'll stop and walk through parts of a challenge together.
* **Important:** use the exact settings given: `test_size=0.2`, `random_state=RANDOM_STATE` in every split and every model that takes one, `cv=5`, `scoring='neg_root_mean_squared_error'`, and the features, models and parameter grids in each challenge. These produce the exact numbers needed for the quiz. Steps marked **📝** feed directly into quiz questions.

**The Professional ML Workflow (5 Stages):**
1. **Data Preparation** — Feature selection, encoding, handling missing values
2. **Initial Train/Test Split** — Lock away your test set (don't touch until Stage 5!)
3. **Model Comparison with CV** — Compare different model types using cross-validation
4. **Hyperparameter Tuning** — Optimize your chosen model with GridSearchCV
5. **Final Test Evaluation** — Evaluate your final model on the locked test set (ONCE!)

### Challenge 1 — Compare Model Types

🧩 **Starter code provided:** fill in the `___` blanks.

**Business Context:** Your real estate company wants to choose the best model architecture for predicting house prices. They're considering three options: Linear Regression, Decision Trees, and Random Forests.

**Your Task:**
1. **Split the data**: Using these six numerical features (`GrLivArea`, `YearBuilt`, `TotalBsmtSF`, `GarageCars`, `FullBath`, `OverallQual`), create an 80/20 train/test split with `random_state=RANDOM_STATE`.
2. **Compare three models** with 5-fold cross-validation on the **training data only** (`scoring='neg_root_mean_squared_error'`):
   - `LinearRegression()`
   - `DecisionTreeRegressor(max_depth=10, random_state=RANDOM_STATE)`
   - `RandomForestRegressor(n_estimators=100, max_depth=10, random_state=RANDOM_STATE)`
3. **Report**: 📝 Record the mean CV RMSE for each model.
4. **Interpret**: 📝 Which model performs best? Why do you think that is?

In [ ]:
# Step 1: Select the features and target (PROVIDED), then split 80/20
features = ['GrLivArea', 'YearBuilt', 'TotalBsmtSF', 'GarageCars', 'FullBath', 'OverallQual']
X = ames[features]
y = ames['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(
    ___, ___, test_size=___, random_state=___
)

print(f"Training set: {len(X_train)} homes")
print(f"Test set:     {len(X_test)} homes (🔒 locked)")

In [ ]:
# Step 2: Compare the three models with 5-fold cross-validation on the training data
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=___, random_state=RANDOM_STATE),
    'Random Forest': RandomForestRegressor(n_estimators=___, max_depth=___, random_state=RANDOM_STATE),
}

cv_results = []
for name, model in models.items():
    scores = cross_val_score(
        model, ___, ___,
        cv=___,
        scoring='neg_root_mean_squared_error'
    )
    cv_results.append({'model': name, 'cv_rmse': -scores.mean()})

# Step 3: Report the mean CV RMSE for each model 📝
cv_results = pd.DataFrame(cv_results)
display(cv_results.style.format({'cv_rmse': '${:,.0f}'}))

# Save the random forest's CV RMSE for Challenge 2 (PROVIDED)
rf_cv_rmse = cv_results.loc[cv_results['model'] == 'Random Forest', 'cv_rmse'].item()

*Step 4 — Your interpretation:*

### Challenge 2 — Systematic Hyperparameter Tuning

🧩 **Starter code provided:** fill in the `___` blanks.

**Business Context:** Now that you've identified which model type works best, your manager wants you to find the best hyperparameter settings for that model.

**Your Task:**
1. **Define the grid**: Use Challenge 1's training data and tune a `RandomForestRegressor(random_state=RANDOM_STATE)` over:
    - `n_estimators`: [100, 200, 300]
    - `max_depth`: [5, 10, 15, 20]
    - `min_samples_split`: [2, 5, 10]
    - `min_samples_leaf`: [1, 2, 5]
2. **Search**: Run `GridSearchCV` with `cv=5`, `scoring='neg_root_mean_squared_error'`, and `n_jobs=-1`.
3. **Report**: 📝 Record the best parameters and the best CV RMSE.
4. **Compare**: 📝 Did tuning improve on Challenge 1's random forest (`n_estimators=100`, `max_depth=10`)? If so, by how much?

> ⏳ **This cell takes a while.** The grid has 3 × 4 × 3 × 3 = 108 combinations, and each one is fit on 5 folds. `n_jobs=-1` spreads the work across all of your computer's CPU cores, but expect it to take **3–5 minutes** in Colab. While it runs, the cell shows `[*]`; it hasn't frozen, so don't rerun it.

In [ ]:
# Step 1: Define the parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': ___,
    'min_samples_split': ___,
    'min_samples_leaf': ___,
}

# Step 2: Run GridSearchCV on the training data (uses all CPU cores)
rf = RandomForestRegressor(random_state=___)
grid_search = GridSearchCV(
    rf,
    ___,
    cv=___,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(___, ___)

# Step 3: Report the best parameters and best CV RMSE 📝
best_params = grid_search.___
best_cv_rmse = -grid_search.___
print(f"Best parameters: {best_params}")
print(f"Best CV RMSE: ${best_cv_rmse:,.0f}")

# Step 4: Compare with Challenge 1's random forest 📝
print(f"Improvement over Challenge 1's forest: ${rf_cv_rmse - best_cv_rmse:,.0f}")

*Step 4 — Your answer:*

### Challenge 3 — Build a Complete Pipeline

📝 **Outline only:** the code cell lists each step as a comment. You write all the code.

**Business Context:** Your model is performing well, but you think you can do better by adding more features and some feature engineering. Let's do this safely by using a `Pipeline()`, so there's no data leakage and the workflow is easy to reproduce.

**Your Task:**
1. **Select features**: Start from the original `ames` data with these features:
   - Numeric: `GrLivArea`, `YearBuilt`, `TotalBsmtSF`, `GarageCars`, `FullBath`, `OverallQual`, `YearRemodAdd`, `BedroomAbvGr`, `TotRmsAbvGrd`
   - Categorical: `Neighborhood`, `HouseStyle`
2. **Split the data**: 80/20 train/test split with `random_state=RANDOM_STATE`.
3. **Build a pipeline** with two steps named `'preprocessor'` and `'model'`:
   - `'preprocessor'`: a `ColumnTransformer` that applies `StandardScaler()` to the numeric features first, then `OneHotEncoder(handle_unknown='ignore')` to the categorical features
   - `'model'`: a `RandomForestRegressor` with Challenge 2's best parameters and `random_state=RANDOM_STATE`
4. **Cross-validate**: 5-fold CV on the training data with `scoring='neg_root_mean_squared_error'`. 📝 Record the mean CV RMSE.
5. **Compare**: 📝 Did the extra features and feature engineering improve on Challenge 2's best CV RMSE? If so, by how much? What do you think explains the change?

In [ ]:
# Step 1: Select the features (PROVIDED) and the target
numeric_features = ['GrLivArea', 'YearBuilt', 'TotalBsmtSF', 'GarageCars', 'FullBath', 'OverallQual',
                    'YearRemodAdd', 'BedroomAbvGr', 'TotRmsAbvGrd']
categorical_features = ['Neighborhood', 'HouseStyle']

# Step 2: Split X = ames[numeric_features + categorical_features] and y = ames['SalePrice']
#         80/20 with random_state=RANDOM_STATE

# Step 3: Build the ColumnTransformer (numeric step first, then categorical), then a
#         Pipeline with steps 'preprocessor' and 'model'. For the model, use
#         RandomForestRegressor(**best_params, random_state=RANDOM_STATE)
#         (**best_params unpacks Challenge 2's best settings into keyword arguments)

# Step 4: 5-fold CV of the pipeline on the training data; print the mean CV RMSE 📝

# Step 5: Compare with best_cv_rmse from Challenge 2 📝


*Step 5 — Your answer:*

### Challenge 4 — Complete End-to-End Workflow

📝 **Outline only:** the code cell lists each step as a comment. You write all the code.

**Business Context:** Your manager wants you to run the complete professional ML workflow from start to finish, using the same feature set as Challenge 3. This time, instead of reusing Challenge 2's settings, you'll tune the forest **inside** the pipeline.

**Your Task:**
1. **Data preparation & split**: Same features, target, and 80/20 split (`random_state=RANDOM_STATE`) as Challenge 3. Lock away the test set.
2. **Pipeline**: The same `'preprocessor'` as Challenge 3, followed by a `'model'` step with `RandomForestRegressor(random_state=RANDOM_STATE)`.
3. **Hyperparameter tuning**: Run `GridSearchCV` on the pipeline with `cv=5`, `scoring='neg_root_mean_squared_error'`, and `n_jobs=-1`, over the same values as Challenge 2:
    - `n_estimators`: [100, 200, 300]
    - `max_depth`: [5, 10, 15, 20]
    - `min_samples_split`: [2, 5, 10]
    - `min_samples_leaf`: [1, 2, 5]

   📝 The grid keys need a prefix, like `'model__n_estimators'`. Why?
4. **Final evaluation**: `GridSearchCV` automatically retrains the best pipeline on the full training set. Save it as **`best_model`** (`grid_search.best_estimator_`); you'll use it in Challenges 5 and 6. Evaluate it **once** on the test set. 📝 Record the best CV RMSE and the test RMSE. Why might they differ?
5. **Reflect**: 📝 How many models did this grid search train in total? 📝 Which parts of this workflow prevent data leakage?

> ⏳ **This cell takes a while,** about twice as long as Challenge 2's. Expect **6–10 minutes** in Colab with `n_jobs=-1`.

In [ ]:
# Step 1: Select the features (PROVIDED) and the target, then split 80/20 with random_state=RANDOM_STATE
numeric_features = ['GrLivArea', 'YearBuilt', 'TotalBsmtSF', 'GarageCars', 'FullBath', 'OverallQual',
                    'YearRemodAdd', 'BedroomAbvGr', 'TotRmsAbvGrd']
categorical_features = ['Neighborhood', 'HouseStyle']

# Step 2: Build the same ColumnTransformer as Challenge 3, then a Pipeline with steps
#         'preprocessor' and 'model', where 'model' is RandomForestRegressor(random_state=RANDOM_STATE)

# Step 3: Define the parameter grid with 'model__' prefixes, then run GridSearchCV
#         (cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1) on the training data

# Step 4: Print the best parameters and best CV RMSE 📝. Save grid_search.best_estimator_
#         as best_model, predict on X_test ONCE, and print the test RMSE
#         with root_mean_squared_error 📝


*Steps 3–5 — Your answers:*

### Challenge 5 — Model Interpretation: Permutation Importance

🚀 **On your own:** no starter code. Use `best_model`, `X_test`, and `y_test` from Challenge 4.

**Business Context:** Your manager wants to understand *why* your model makes its predictions. Which features are most influential for predicting house prices?

**Your Task:**
1. **Permutation importance**: Calculate permutation importance for `best_model` on the **test** set with `scoring='neg_root_mean_squared_error'`, `n_repeats=10`, and `random_state=RANDOM_STATE`. Because `best_model` is a pipeline, the importances are for the 11 original columns of `X_test`, not the one-hot columns.
2. **Visualize**: Create a horizontal bar chart of the mean importance for all 11 features, sorted from most to least important. Use matplotlib, seaborn, or any other library.
3. **Interpret**: 📝 Which feature is the most important for your model? What does a high permutation importance tell you?

💡 **Hint:** Revisit Challenge 3 of the Week 11 Lab and Chapter 27. With this `scoring`, each importance is how much the RMSE (in dollars) increases when that feature is shuffled.

In [ ]:
# Your code here

*Step 3 — Your answer:*

### Challenge 6 — Partial Dependence Plot (PDP)

🚀 **On your own:** no starter code. Use `best_model` and `X_test` from Challenge 4.

**Business Context:** Now that you've identified the most important feature, your manager wants to understand *how* it influences the model's predictions. A Partial Dependence Plot (PDP) shows how the predicted sale price changes as this one feature changes, averaging over the values of all the other features.

**Your Task:**
1. **Plot**: Create a PDP for the most important feature from Challenge 5, using `best_model` and the test set.
2. **Interpret**: 📝 How do changes in this feature affect the predicted house price? Is the relationship positive or negative? Linear, or more complex?
3. **Explain it to your manager**: In 2–3 sentences, explain what the PDP shows. Add one caution about what it does *not* tell them.

💡 **Hint:** `PartialDependenceDisplay.from_estimator()` works directly on a pipeline: pass the original column name. If scikit-learn warns (or, in newer versions, errors) that the column "contains integer data", pass a copy of the test set with that column converted to float, for example `X_test.astype({'YourFeature': float})`.

In [ ]:
# Your code here

*Steps 2–3 — Your answers:*

## 🎓 Lab Wrap-Up & Reflection

### ✅ What You Accomplished
In this lab, you practiced:
- Comparing models and settings with cross-validation, reading both the mean and the std
- Tuning hyperparameters systematically with `GridSearchCV`
- Scaling and encoding features inside a leak-free `ColumnTransformer` + `Pipeline`
- Running the complete professional workflow: split, make every decision with CV, refit, test once
- Explaining your final model with permutation importance and a partial dependence plot

### 🤔 Reflection Questions
Take 2–3 minutes to consider:
- Your test RMSE in Challenge 4 didn't exactly match your best CV RMSE. Why is that expected, and why shouldn't you re-tune after seeing it?
- The grid searches in Challenges 2 and 4 took several minutes. How would you decide which hyperparameters and values are worth searching on a bigger project?
- Where in your past projects did you make a decision using test-set results? How would you redo it with cross-validation?

### 🔗 Connection to Course Goals
This workflow is what turns a model you built into a number you can stand behind. Every decision now happens on the training data, and the test set gives one honest, final answer. You'll use this same workflow, pipeline and all, for every model you build from here on.

### 📋 Next Steps
- **Homework:** Complete the Canvas quiz using your numerical results from today's challenges, and upload your completed notebook (due Sunday)
- **Next Tuesday:** Unsupervised learning: finding structure in data when there's no target to predict
- **Additional Practice:** Try the end-of-chapter exercises in Chapters 28–30

---
**💾 Save your work.** You'll upload this notebook with your quiz, so make sure every cell has been run and shows its output.

## 🚨 Troubleshooting & Common Issues

**Issue 1:** My grid search cell has been running for minutes. Is it frozen?
- **Solution:** Probably not. Challenges 2 and 4 each train 540 models; expect several minutes in Colab. While `[*]` shows, it's still working. Don't rerun the cell, which only starts it over.

**Issue 2:** My numbers don't match my groupmates'
- **Solution:** Check that you used `test_size=0.2` and `random_state=RANDOM_STATE` in the split and in every model that takes one, `cv=5`, and `scoring='neg_root_mean_squared_error'`. In Challenges 3–4, list the numeric step before the categorical step in the `ColumnTransformer`.

**Issue 3:** My RMSE is negative
- **Solution:** scikit-learn returns `neg_root_mean_squared_error` so that higher is always better. Flip the sign: `-scores.mean()` or `-grid_search.best_score_`.

**Issue 4:** `ValueError: Invalid parameter 'n_estimators' for estimator Pipeline`
- **Solution:** Inside a pipeline, prefix each grid key with the step name and two underscores: `'model__n_estimators'`, not `'n_estimators'`.

**Issue 5:** `ValueError: could not convert string to float: 'NAmes'`
- **Solution:** A text column reached a model without being encoded. This usually happens when you go back and rerun an earlier cell after a later one replaced `X_train` with text columns: a Part A 🔧 Try it cell after Step 4, or Challenge 1–2 after Challenge 3. Rerun that section's split cell first, or restart and run the notebook in order.

**Issue 6:** `ValueError: Found unknown categories` during cross-validation
- **Solution:** Use `OneHotEncoder(handle_unknown='ignore')`, so a category that appears only in a validation fold or the test set becomes all zeros instead of an error.

**Issue 7:** Some permutation importances are slightly negative
- **Solution:** That's normal. Shuffling an unimportant feature can improve performance slightly by chance. Treat those features as unimportant.